# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(metadata)

# For display: show the dataset's title and description
print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will print available record set `@id`s, and then for each record set, list its field `@id`s and columns.

In [ ]:
# Collect all RecordSet @ids
record_sets = []
if hasattr(metadata, 'record_set') and metadata.record_set:
    for rs in metadata.record_set:
        if hasattr(rs, '@id'):
            record_sets.append(rs['@id'])
        elif hasattr(rs, 'id'):
            record_sets.append(rs.id)

if not record_sets:
    # Some croissant schemas keep record_sets under 'recordSet'
    if hasattr(metadata, 'recordSet') and metadata.recordSet:
        for rs in metadata.recordSet:
            if isinstance(rs, dict) and '@id' in rs:
                record_sets.append(rs['@id'])
            elif hasattr(rs, '@id'):
                record_sets.append(rs.@id)

if not record_sets:
    print('No record sets found in schema metadata!')
else:
    print('Available Record Sets:')
    for i, rsid in enumerate(record_sets):
        print(f"  {i+1}. @id: {rsid}")

    # For each record set, print their field and column @ids.
    for rsid in record_sets:
        print(f"\nFields in RecordSet {rsid}:")
        rs_obj = None
        # Find record set object
        if hasattr(metadata, 'record_set'):
            for rs in metadata.record_set:
                if (isinstance(rs, dict) and rs.get('@id') == rsid) or (hasattr(rs, '@id') and rs.@id == rsid):
                    rs_obj = rs
                    break
        elif hasattr(metadata, 'recordSet'):
            for rs in metadata.recordSet:
                if (isinstance(rs, dict) and rs.get('@id') == rsid) or (hasattr(rs, '@id') and rs.@id == rsid):
                    rs_obj = rs
                    break

        # Print fields and columns
        if rs_obj is not None:
            fields = getattr(rs_obj, 'field', None) or getattr(rs_obj, 'fields', None) or rs_obj.get('field', []) if isinstance(rs_obj, dict) else []
            columns = getattr(rs_obj, 'column', None) or getattr(rs_obj, 'columns', None) or rs_obj.get('column', []) if isinstance(rs_obj, dict) else []

            print('  Field @ids:')
            for f in fields if fields else []:
                if isinstance(f, dict) and '@id' in f:
                    print('    -', f['@id'])
                elif hasattr(f, '@id'):
                    print('    -', f.@id)
                else:
                    print('    -', str(f))

            print('  Column @ids:')
            for c in columns if columns else []:
                if isinstance(c, dict) and '@id' in c:
                    print('    -', c['@id'])
                elif hasattr(c, '@id'):
                    print('    -', c.@id)
                else:
                    print('    -', str(c))
        else:
            print('  No fields or columns found.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Note:** Only record sets and columns/fields referenced by their `@id` are used.

In [ ]:
# If you already discovered the main record set's @id in the previous cell, assign it here.
# Otherwise, you can find it by inspecting metadata in the previous cell or by examining the schema directly.

# For this dataset, let's define the list manually:
# Assume the record set corresponding to the main data table is 'https://api.app.sen.science/frontiers/7154140/fair2/proteins'

record_sets = ['https://api.app.sen.science/frontiers/7154140/fair2/proteins']
dataframes = {}

for record_set in record_sets:
    print(f"\nLoading data for record set: {record_set}")
    records = list(dataset.records(record_set=record_set))
    if records:
        dataframes[record_set] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records with columns:", dataframes[record_set].columns.tolist())
    else:
        print("No records found for this record set!")

main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/fair2/proteins'

if main_record_set_id in dataframes:
    print(dataframes[main_record_set_id].head())
else:
    print(f"DataFrame for {main_record_set_id} not available.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

All column references are via their `@id` as provided in the Croissant schema.

In [ ]:
# Analyze a numeric field, e.g., 'protein_coverage' with its actual column ID
# Replace with the correct @id for the numeric field in your dataset schema
main_df = dataframes.get(main_record_set_id, pd.DataFrame())

# Example field IDs (these must be adapted to match exact @id from the schema; placeholders used here):
numeric_field = 'https://api.app.sen.science/frontiers/7154140/fair2/proteins/fields/cov'  # Example: protein_coverage
group_field = 'https://api.app.sen.science/frontiers/7154140/fair2/proteins/fields/description'  # Example: protein description

real_numeric_field_col = None
# Try to match the DataFrame column
for col in main_df.columns:
    if numeric_field in col:
        real_numeric_field_col = col
        break
    elif col.endswith(numeric_field.split('/')[-1]):
        real_numeric_field_col = col
        break
if real_numeric_field_col is None:
    print(f"Numeric field @id {numeric_field} not found in columns!")
else:
    threshold = 10
    filtered_df = main_df[main_df[real_numeric_field_col].astype(float) > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold} (actually column '{real_numeric_field_col}'):")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{real_numeric_field_col}_normalized"] = (filtered_df[real_numeric_field_col] - filtered_df[real_numeric_field_col].mean()) / filtered_df[real_numeric_field_col].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[real_numeric_field_col, f"{real_numeric_field_col}_normalized"]].head())

    # Grouping (if group_field exists)
    real_group_field_col = None
    for col in main_df.columns:
        if group_field in col:
            real_group_field_col = col
            break
        elif col.endswith(group_field.split('/')[-1]):
            real_group_field_col = col
            break

    if real_group_field_col and real_group_field_col in filtered_df:
        grouped_df = filtered_df.groupby(real_group_field_col).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (column '{real_group_field_col}'):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt

# Histogram of the coverage field (if loaded)
if real_numeric_field_col and real_numeric_field_col in main_df:
    plt.figure(figsize=(8,4))
    main_df[real_numeric_field_col].astype(float).hist(bins=30, color='skyblue')
    plt.title('Distribution of Protein Coverage')
    plt.xlabel('Coverage (%)')
    plt.ylabel('Count')
    plt.show()

# If another numeric field exists, plot a scatter
second_numeric_field = None
for col in main_df.columns:
    if col != real_numeric_field_col and main_df[col].dtype != object:
        second_numeric_field = col
        break

if second_numeric_field:
    plt.figure(figsize=(6,4))
    plt.scatter(main_df[real_numeric_field_col], main_df[second_numeric_field], s=20, alpha=0.6)
    plt.xlabel(real_numeric_field_col)
    plt.ylabel(second_numeric_field)
    plt.title(f"Scatter plot: {real_numeric_field_col} vs {second_numeric_field}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


In this notebook, we demonstrated how to use the `mlcroissant` library to discover and load data defined by a Croissant schema. We performed initial exploratory analysis of the protein record set, exploring numeric field distributions and grouping by attributes. Further analyses could include deeper statistical analysis, additional filtering, or integration with domain knowledge for mass spectrometry data.